# YouTube AI Studio v7 - Generador Visual

Notebook automatizado para Kaggle (GPU T4 x2).

**Flujo:**
1. Lee `prompts.json` desde el dataset `ytcreator-prompts`
2. Genera imagenes con Juggernaut XL (SDXL)
3. Genera video con Wan 2.1 para escenas marcadas con `usa_video_ia`
4. Aplica zoom/pan cinematico a escenas sin video IA
5. Guarda todo en `/kaggle/working/` (output automatico)

In [ ]:
# Celda 1 - Instalar dependencias
!pip install -q diffusers transformers accelerate safetensors
!pip install -q opencv-python-headless pillow imageio

In [ ]:
# Celda 2 - Leer prompts.json desde el dataset de Kaggle
import json
from pathlib import Path

DATASET_PATH = Path("/kaggle/input/ytcreator-prompts/prompts.json")
OUTPUT_DIR = Path("/kaggle/working")

if not DATASET_PATH.exists():
    raise FileNotFoundError(
        f"No se encontro {DATASET_PATH}. "
        "Asegurate de que el dataset 'ytcreator-prompts' este agregado como input."
    )

with open(DATASET_PATH, "r", encoding="utf-8") as f:
    data = json.load(f)

PROYECTO_ID = data["proyecto_id"]
ESTILO_APLICADO = data.get("estilo_aplicado", "sin_estilo")
ESCENAS = data["escenas"]

escenas_imagen = [e for e in ESCENAS]
escenas_video = [e for e in ESCENAS if e.get("usa_video_ia", False)]
escenas_zoom = [e for e in ESCENAS if not e.get("usa_video_ia", False)]

print(f"Proyecto: {PROYECTO_ID}")
print(f"Estilo aplicado: {ESTILO_APLICADO}")
print(f"Total escenas: {len(ESCENAS)}")
print(f"  - Con video IA (Wan 2.1): {len(escenas_video)}")
print(f"  - Con zoom/pan: {len(escenas_zoom)}")
print(f"\nPrimer prompt: {ESCENAS[0]['prompt'][:120]}...")
if ESCENAS[0].get("negative_prompt"):
    print(f"Primer negative: {ESCENAS[0]['negative_prompt'][:80]}...")

In [ ]:
# Celda 3 - Cargar modelo de imagenes (Juggernaut XL)
import torch
from diffusers import StableDiffusionXLPipeline, DPMSolverMultistepScheduler

MODEL_ID = "RunDiffusion/Juggernaut-XL-v9"

pipe_img = StableDiffusionXLPipeline.from_pretrained(
    MODEL_ID,
    torch_dtype=torch.float16,
    variant="fp16",
    use_safetensors=True,
)
pipe_img.scheduler = DPMSolverMultistepScheduler.from_config(pipe_img.scheduler.config)
pipe_img = pipe_img.to("cuda")
pipe_img.enable_model_cpu_offload()

print(f"Modelo cargado: {MODEL_ID}")
print(f"GPU: {torch.cuda.get_device_name(0)}")
print(f"VRAM: {torch.cuda.get_device_properties(0).total_mem / 1e9:.1f} GB")

In [ ]:
# Celda 4 - Generar imagenes para TODAS las escenas
from IPython.display import display, clear_output
import time

# Fallback si la escena no trae negative_prompt (retrocompatibilidad)
DEFAULT_NEGATIVE_PROMPT = (
    "blurry, low quality, distorted, deformed, ugly, bad anatomy, "
    "bad hands, extra fingers, mutated, watermark, text, logo, "
    "oversaturated, cartoon, 3d render, doll"
)

WIDTH = 1280
HEIGHT = 720
STEPS = 30
CFG = 7.0
SEED_BASE = 42

imagenes_generadas = {}

print(f"Generando {len(ESCENAS)} imagenes...\n")
t0 = time.time()

for i, escena in enumerate(ESCENAS):
    numero = escena["numero"]
    prompt = escena["prompt"]
    negative = escena.get("negative_prompt") or DEFAULT_NEGATIVE_PROMPT
    output_path = OUTPUT_DIR / f"escena_{numero:02d}.png"

    print(f"[{i+1}/{len(ESCENAS)}] Escena {numero}...")

    generator = torch.Generator("cuda").manual_seed(SEED_BASE + numero)

    image = pipe_img(
        prompt=prompt,
        negative_prompt=negative,
        width=WIDTH,
        height=HEIGHT,
        num_inference_steps=STEPS,
        guidance_scale=CFG,
        generator=generator,
    ).images[0]

    image.save(str(output_path))
    imagenes_generadas[numero] = str(output_path)
    elapsed = time.time() - t0
    per_img = elapsed / (i + 1)
    remaining = per_img * (len(ESCENAS) - i - 1)
    print(f"  OK ({per_img:.1f}s/img, ~{remaining/60:.1f}min restantes)")

    torch.cuda.empty_cache()

print(f"\n{len(imagenes_generadas)} imagenes generadas en {(time.time()-t0)/60:.1f} min")

In [ ]:
# Celda 5 - Liberar VRAM del modelo de imagenes antes de cargar video
del pipe_img
torch.cuda.empty_cache()
import gc
gc.collect()
print(f"VRAM liberada. Disponible: {torch.cuda.mem_get_info()[0]/1e9:.1f} GB")

In [ ]:
# Celda 6 - Generar videos con Wan 2.1 (escenas marcadas usa_video_ia=true)
import numpy as np

DEFAULT_VIDEO_NEGATIVE = "static, blurry, distorted"

videos_generados = {}

if escenas_video:
    print(f"Cargando Wan 2.1 para {len(escenas_video)} escenas con video IA...")

    try:
        from diffusers import WanImageToVideoPipeline
        from diffusers.utils import load_image

        pipe_video = WanImageToVideoPipeline.from_pretrained(
            "Wan-AI/Wan2.1-I2V-14B-480P-Diffusers",
            torch_dtype=torch.float16,
        )
        pipe_video.enable_model_cpu_offload()

        t0_vid = time.time()

        for i, escena in enumerate(escenas_video):
            numero = escena["numero"]
            img_path = imagenes_generadas.get(numero)
            if not img_path:
                print(f"  [SKIP] Escena {numero}: no se genero imagen")
                continue

            print(f"[{i+1}/{len(escenas_video)}] Video escena {numero}...")

            image = load_image(img_path).resize((832, 480))

            # Usar negative_prompt de la escena si existe, con fallback
            neg = escena.get("negative_prompt") or DEFAULT_VIDEO_NEGATIVE
            # "static" es critico para video (evita frames congelados)
            if "static" not in neg.lower():
                neg = f"static, {neg}"

            output = pipe_video(
                image=image,
                prompt=escena["prompt"],
                negative_prompt=neg,
                num_inference_steps=20,
                guidance_scale=5.0,
                num_frames=33,
                generator=torch.Generator("cuda").manual_seed(SEED_BASE + numero),
            )

            video_path = OUTPUT_DIR / f"escena_{numero:02d}.mp4"

            from diffusers.utils import export_to_video
            export_to_video(output.frames[0], str(video_path), fps=16)

            videos_generados[numero] = str(video_path)
            print(f"  OK")
            torch.cuda.empty_cache()

        del pipe_video
        torch.cuda.empty_cache()
        gc.collect()
        print(f"\n{len(videos_generados)} videos IA en {(time.time()-t0_vid)/60:.1f} min")

    except Exception as e:
        print(f"Wan 2.1 no disponible ({e}), usando zoom/pan para todas las escenas")
        escenas_zoom = ESCENAS.copy()
        escenas_video = []
else:
    print("No hay escenas marcadas para video IA, todas usaran zoom/pan")

In [ ]:
# Celda 7 - Zoom/pan cinematico para escenas SIN video IA
import cv2
from PIL import Image

ZOOM_DURACION_SEG = 5
ZOOM_FPS = 24
ZOOM_FACTOR = 1.15

MOVIMIENTOS = ["zoom_in", "zoom_out", "pan_left", "pan_right", "pan_up"]


def crear_video_zoom(img_path, output_path, movimiento="zoom_in",
                     duracion=ZOOM_DURACION_SEG, fps=ZOOM_FPS, zoom=ZOOM_FACTOR):
    img = cv2.imread(str(img_path))
    h, w = img.shape[:2]
    total_frames = duracion * fps

    fourcc = cv2.VideoWriter_fourcc(*"mp4v")
    writer = cv2.VideoWriter(str(output_path), fourcc, fps, (w, h))

    for frame_i in range(total_frames):
        t = frame_i / total_frames

        if movimiento == "zoom_in":
            s = 1.0 + (zoom - 1.0) * t
            cx, cy = w / 2, h / 2
        elif movimiento == "zoom_out":
            s = zoom - (zoom - 1.0) * t
            cx, cy = w / 2, h / 2
        elif movimiento == "pan_left":
            s = zoom
            cx = w / 2 + (w * 0.1) * (1 - t)
            cy = h / 2
        elif movimiento == "pan_right":
            s = zoom
            cx = w / 2 - (w * 0.1) * (1 - t)
            cy = h / 2
        elif movimiento == "pan_up":
            s = zoom
            cx = w / 2
            cy = h / 2 + (h * 0.08) * (1 - t)
        else:
            s = 1.0
            cx, cy = w / 2, h / 2

        new_w = int(w / s)
        new_h = int(h / s)
        x1 = max(0, int(cx - new_w / 2))
        y1 = max(0, int(cy - new_h / 2))
        x2 = min(w, x1 + new_w)
        y2 = min(h, y1 + new_h)

        crop = img[y1:y2, x1:x2]
        frame = cv2.resize(crop, (w, h), interpolation=cv2.INTER_LANCZOS4)
        writer.write(frame)

    writer.release()


escenas_sin_video = [e for e in ESCENAS if e["numero"] not in videos_generados]

print(f"Generando zoom/pan para {len(escenas_sin_video)} escenas...\n")

for i, escena in enumerate(escenas_sin_video):
    numero = escena["numero"]
    img_path = imagenes_generadas.get(numero)
    if not img_path:
        print(f"  [SKIP] Escena {numero}: no hay imagen")
        continue

    movimiento = MOVIMIENTOS[i % len(MOVIMIENTOS)]
    video_path = OUTPUT_DIR / f"escena_{numero:02d}.mp4"

    print(f"[{i+1}/{len(escenas_sin_video)}] Escena {numero} ({movimiento})...")
    crear_video_zoom(img_path, video_path, movimiento=movimiento)
    videos_generados[numero] = str(video_path)
    print(f"  OK")

print(f"\nTotal videos generados: {len(videos_generados)}")

In [ ]:
# Celda 8 - Resumen final
print("=" * 50)
print(f"PROYECTO: {PROYECTO_ID}")
print(f"=" * 50)

pngs = sorted(OUTPUT_DIR.glob("escena_*.png"))
mp4s = sorted(OUTPUT_DIR.glob("escena_*.mp4"))

print(f"\nImagenes generadas: {len(pngs)}")
for p in pngs:
    size_kb = p.stat().st_size / 1024
    print(f"  {p.name} ({size_kb:.0f} KB)")

print(f"\nVideos generados: {len(mp4s)}")
for p in mp4s:
    size_mb = p.stat().st_size / (1024 * 1024)
    print(f"  {p.name} ({size_mb:.1f} MB)")

total_mb = sum(p.stat().st_size for p in list(pngs) + list(mp4s)) / (1024 * 1024)
print(f"\nTamano total output: {total_mb:.1f} MB")
print("\nListo para descarga automatica por el agente 3.2")